In [ ]:
# ============================================================
# CUADERNO 03 — ISOLATION FOREST
# Semana 10: Ejecutar Experimentos
# Proyecto: Modelos de ML para Detección de Anomalías
#           en el Tráfico y Logs de Entornos Cloud
# ============================================================
# Respaldo metodológico:
# - Mitropoulou et al. (2024) — hiperparámetros IF
# - Almuhanna & Dardouri (2025) — proporción 70/30
# - Park et al. (2025) — umbral contaminación
# - Aljuaid & Alshamrani (2024) — métricas evaluación
# ============================================================

In [ ]:
# ── CELDA 1: Instalaciones y configuración ──────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install memory-profiler psutil -q

import pandas as pd
import numpy as np
import time
import psutil
import os
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             f1_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

ruta_procesados = '/content/drive/MyDrive/IDS2018_procesados'
ruta_resultados = '/content/drive/MyDrive/IDS2018_resultados'
os.makedirs(ruta_resultados, exist_ok=True)

print("✅ Librerías cargadas correctamente")

Mounted at /content/drive
✅ Librerías cargadas correctamente


In [ ]:
# ── CELDA 2: Cargar datos preprocesados ─────────────────────
print("="*60)
print("CARGANDO DATOS PREPROCESADOS")
print("="*60)

df_train_full = pd.read_csv(f"{ruta_procesados}/dataset_train.csv")
df_test = pd.read_csv(f"{ruta_procesados}/dataset_test.csv")
df_labels = pd.read_csv(f"{ruta_procesados}/dataset_test_labels.csv")

y_test_labels = df_labels.iloc[:, 0]

print(f"Training set completo : {df_train_full.shape[0]:,} filas benignas")
print(f"Test set              : {df_test.shape[0]:,} filas")
print(f"\nDistribucion test set:")
print(y_test_labels.value_counts().to_string())

CARGANDO DATOS PREPROCESADOS
Training set completo : 3,805,770 filas benignas
Test set              : 4,967,041 filas

Distribucion test set:
Label
Benign                   3805770
DDOS attack-HOIC          668461
Bot                       282310
SSH-Bruteforce            117322
DoS attacks-GoldenEye      41455
FTP-BruteForce             39346
DoS attacks-Slowloris      10285
DDOS attack-LOIC-UDP        1730
Brute Force -Web             249
Brute Force -XSS              79
SQL Injection                 34


In [ ]:
# ── CELDA 3: Preparar partición 70/30 ───────────────────────
# Respaldo: Almuhanna & Dardouri (2025) — proporción 70/30
# Training: 70% de filas benignas
# Test: 30% filas benignas (sobrantes) + TODOS los ataques
print("="*60)
print("PREPARACION PARTICION 70/30")
print("="*60)

# Convertir etiquetas a binario: 0=Benigno, 1=Ataque
y_test_binary = (y_test_labels != 'Benign').astype(int)

total_benignos = df_train_full.shape[0]
n_train = int(total_benignos * 0.70)

print(f"Total benignos disponibles : {total_benignos:,}")
print(f"Training set (70%)         : {n_train:,} filas")
print(f"Test set benignos (30%)    : {total_benignos - n_train:,} filas")
print(f"Test set ataques           : {y_test_binary.sum():,} filas")
print(f"Test set total             : {df_test.shape[0]:,} filas")

PREPARACION PARTICION 70/30
Total benignos disponibles : 3,805,770
Training set (70%)         : 2,664,039 filas
Test set benignos (30%)    : 1,141,731 filas
Test set ataques           : 1,161,271 filas
Test set total             : 4,967,041 filas


In [ ]:
# ── CELDA 3B: Eliminar columnas no numéricas ─────────────────
print("Columnas no numéricas encontradas:")
cols_no_numericas = df_train_full.select_dtypes(exclude=[np.number]).columns.tolist()
print(cols_no_numericas)

# Eliminar de train y test
df_train_full = df_train_full.select_dtypes(include=[np.number])
df_test = df_test.select_dtypes(include=[np.number])

print(f"\nDimensiones corregidas:")
print(f"Training set: {df_train_full.shape}")
print(f"Test set    : {df_test.shape}")
print("✅ Listo — ejecuta la Celda 4 nuevamente")

Columnas no numéricas encontradas:
['Timestamp']

Dimensiones corregidas:
Training set: (3805770, 13)
Test set    : (4967041, 13)
✅ Listo — ejecuta la Celda 4 nuevamente


In [ ]:
# ── CELDA 4: Experimento con 5 repeticiones ─────────────────
# Respaldo: 5 semillas aleatorias diferentes
# Almuhanna & Dardouri (2025)
print("="*60)
print("ISOLATION FOREST — 5 REPETICIONES")
print("="*60)
print("Hiperparametros (Mitropoulou et al., 2024):")
print("  n_estimators  : 100")
print("  contamination : 0.1")
print("  max_samples   : min(256, n)")
print("  max_features  : 1")
print("="*60)

# Almacenar resultados de cada repetición
resultados = []
semillas = [42, 123, 456, 789, 1024]

for i, semilla in enumerate(semillas):
    print(f"\nRepeticion {i+1}/5 (semilla={semilla})...")

    # Muestra 70% benigno para training
    np.random.seed(semilla)
    idx_train = np.random.choice(
        df_train_full.index,
        size=n_train,
        replace=False
    )
    X_train = df_train_full.loc[idx_train].values
    X_test = df_test.values

    # ── Medir tiempo de entrenamiento ──
    proceso = psutil.Process(os.getpid())
    mem_antes = proceso.memory_info().rss / 1024 / 1024
    cpu_antes = psutil.cpu_percent(interval=0.1)

    t_inicio = time.time()

    modelo = IsolationForest(
        n_estimators=100,
        contamination=0.1,
        max_samples=min(256, X_train.shape[0]),
        max_features=1,
        random_state=semilla,
        n_jobs=-1
    )
    modelo.fit(X_train)

    t_fin_entren = time.time()
    tiempo_entrenamiento = t_fin_entren - t_inicio

    # ── Medir velocidad de inferencia ──
    t_inicio_inf = time.time()
    y_pred_raw = modelo.predict(X_test)
    t_fin_inf = time.time()
    tiempo_inferencia = t_fin_inf - t_inicio_inf

    # Medir recursos
    mem_despues = proceso.memory_info().rss / 1024 / 1024
    cpu_uso = psutil.cpu_percent(interval=0.1)

    # Isolation Forest: -1=anomalía, 1=normal → convertir a 0/1
    y_pred = (y_pred_raw == -1).astype(int)

    # ── Calcular anomaly score ──
    anomaly_scores = -modelo.score_samples(X_test)

    # ── Métricas VD ──
    acc = accuracy_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred, zero_division=0)
    prec = precision_score(y_test_binary, y_pred, zero_division=0)
    rec = recall_score(y_test_binary, y_pred, zero_division=0)

    # FPR
    cm = confusion_matrix(y_test_binary, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    # AUC-ROC
    try:
        auc = roc_auc_score(y_test_binary, anomaly_scores)
    except:
        auc = 0.0

    # ── Métricas VI ──
    vel_inferencia = X_test.shape[0] / tiempo_inferencia
    uso_memoria = mem_despues - mem_antes
    uso_cpu = cpu_uso

    resultados.append({
        "Repeticion": i + 1,
        "Semilla": semilla,
        # VI
        "Tiempo_entrenamiento_s": round(tiempo_entrenamiento, 4),
        "Velocidad_inferencia_mps": round(vel_inferencia, 2),
        "Uso_memoria_MB": round(uso_memoria, 2),
        "Uso_CPU_pct": round(uso_cpu, 2),
        # VD
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "FPR": round(fpr, 4),
        "AUC_ROC": round(auc, 4),
        # Anomaly score stats
        "Score_medio_normal": round(
            anomaly_scores[y_test_binary == 0].mean(), 4),
        "Score_medio_ataque": round(
            anomaly_scores[y_test_binary == 1].mean(), 4),
    })

    print(f"  ✅ Tiempo entren.   : {tiempo_entrenamiento:.2f} s")
    print(f"  ✅ Vel. inferencia  : {vel_inferencia:,.0f} muestras/s")
    print(f"  ✅ Uso memoria      : {uso_memoria:.2f} MB")
    print(f"  ✅ F1-Score         : {f1:.4f}")
    print(f"  ✅ AUC-ROC          : {auc:.4f}")

print("\n✅ 5 repeticiones completadas")

ISOLATION FOREST — 5 REPETICIONES
Hiperparametros (Mitropoulou et al., 2024):
  n_estimators  : 100
  contamination : 0.1
  max_samples   : min(256, n)
  max_features  : 1

Repeticion 1/5 (semilla=42)...
  ✅ Tiempo entren.   : 22.75 s
  ✅ Vel. inferencia  : 216,661 muestras/s
  ✅ Uso memoria      : 120.34 MB
  ✅ F1-Score         : 0.2184
  ✅ AUC-ROC          : 0.4675

Repeticion 2/5 (semilla=123)...
  ✅ Tiempo entren.   : 16.33 s
  ✅ Vel. inferencia  : 207,831 muestras/s
  ✅ Uso memoria      : 101.68 MB
  ✅ F1-Score         : 0.1732
  ✅ AUC-ROC          : 0.4570

Repeticion 3/5 (semilla=456)...
  ✅ Tiempo entren.   : 17.94 s
  ✅ Vel. inferencia  : 212,854 muestras/s
  ✅ Uso memoria      : 40.94 MB
  ✅ F1-Score         : 0.2211
  ✅ AUC-ROC          : 0.5004

Repeticion 4/5 (semilla=789)...
  ✅ Tiempo entren.   : 16.69 s
  ✅ Vel. inferencia  : 204,155 muestras/s
  ✅ Uso memoria      : 40.68 MB
  ✅ F1-Score         : 0.2206
  ✅ AUC-ROC          : 0.4511

Repeticion 5/5 (semilla=1024)...
 

In [ ]:
# ── CELDA 5: Resultados y estadísticas ──────────────────────
print("="*60)
print("RESULTADOS — ISOLATION FOREST")
print("="*60)

df_resultados = pd.DataFrame(resultados)

# Calcular media y desviación estándar
metricas = [
    "Tiempo_entrenamiento_s",
    "Velocidad_inferencia_mps",
    "Uso_memoria_MB",
    "Uso_CPU_pct",
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score",
    "FPR",
    "AUC_ROC"
]

print("\nResultados por repeticion:")
print(df_resultados[["Repeticion"] + metricas].to_string(index=False))

print("\n" + "="*60)
print("MEDIA ± DESVIACION ESTANDAR")
print("="*60)

resumen = {}
for m in metricas:
    media = df_resultados[m].mean()
    std = df_resultados[m].std()
    resumen[m] = f"{media:.4f} ± {std:.4f}"
    print(f"{m:35s}: {media:.4f} ± {std:.4f}")

RESULTADOS — ISOLATION FOREST

Resultados por repeticion:
 Repeticion  Tiempo_entrenamiento_s  Velocidad_inferencia_mps  Uso_memoria_MB  Uso_CPU_pct  Accuracy  Precision  Recall  F1_Score    FPR  AUC_ROC
          1                 22.7544                 216660.92          120.34          0.0    0.7276     0.3318  0.1628    0.2184 0.1000   0.4675
          2                 16.3255                 207830.82          101.68         70.0    0.7191     0.2776  0.1258    0.1732 0.0999   0.4570
          3                 17.9444                 212853.74           40.94          0.0    0.7281     0.3347  0.1650    0.2211 0.1001   0.5004
          4                 16.6937                 204154.97           40.68          5.0    0.7280     0.3343  0.1647    0.2206 0.1001   0.4511
          5                 16.8126                 221749.80          -55.67          0.0    0.7138     0.2405  0.1039    0.1451 0.1001   0.5181

MEDIA ± DESVIACION ESTANDAR
Tiempo_entrenamiento_s             : 

In [ ]:
# ── CELDA 6: Guardar resultados ──────────────────────────────
print("="*60)
print("GUARDANDO RESULTADOS")
print("="*60)

# Guardar resultados detallados
ruta_csv = f"{ruta_resultados}/IF_resultados_detallados.csv"
df_resultados.to_csv(ruta_csv, index=False)
print(f"✅ Resultados detallados: {ruta_csv}")

# Guardar resumen
resumen_df = pd.DataFrame({
    "Metrica": list(resumen.keys()),
    "Media_±_Std": list(resumen.values())
})
ruta_resumen = f"{ruta_resultados}/IF_resumen.csv"
resumen_df.to_csv(ruta_resumen, index=False)
print(f"✅ Resumen guardado: {ruta_resumen}")

print("\n" + "="*60)
print("ISOLATION FOREST COMPLETADO")
print("="*60)
print(f"Archivos guardados en: {ruta_resultados}")
print("\nSiguiente paso: Cuaderno 04 — Autoencoder")

GUARDANDO RESULTADOS
✅ Resultados detallados: /content/drive/MyDrive/IDS2018_resultados/IF_resultados_detallados.csv
✅ Resumen guardado: /content/drive/MyDrive/IDS2018_resultados/IF_resumen.csv

ISOLATION FOREST COMPLETADO
Archivos guardados en: /content/drive/MyDrive/IDS2018_resultados

Siguiente paso: Cuaderno 04 — Autoencoder
